# Model Training & Quantum Optimization

## QT-2.21 Quantum-Enhanced Medical Imaging for Cancer Detection

---

This notebook provides a comprehensive, end-to-end walkthrough of the **model training pipeline** for the Quantum-Enhanced Medical Imaging project. We train and evaluate **6 distinct model configurations** — 3 classical baselines and 3 quantum-optimized variants — to systematically measure the impact of quantum-inspired optimization on medical image classification.

### The 6 Configurations at a Glance

| # | Name | Feature Selection | Hyperparameter Tuning | Classifier |
|---|------|-------------------|-----------------------|------------|
| 1 | **Baseline A** | Mutual Information (top-k) | GridSearchCV | SVM (RBF) |
| 2 | **Baseline B** | SelectKBest (ANOVA F) | Optuna TPE | XGBoost |
| 3 | **Baseline C** | N/A (end-to-end CNN) | Manual LR schedule | Fine-tuned ResNet50 |
| 4 | **Quantum A** | QIEO Feature Selection | GridSearchCV | SVM (RBF) |
| 5 | **Quantum B** | QIEO Feature Selection | QIEO Hyperparam Opt | XGBoost |
| 6 | **Quantum C** | N/A (end-to-end CNN) | N/A (same as Baseline C) | Fine-tuned ResNet50 |

**Why 6 configurations?**  
By holding the classifier constant and swapping only the optimization strategy (classical → quantum), we can isolate the contribution of quantum-inspired algorithms. Configurations 1 vs 4 test quantum feature selection; configurations 2 vs 5 test quantum hyperparameter optimization; and configuration 3/6 provides a deep-learning reference point.

### Key Concepts

- **QIEO (Quantum-Inspired Evolutionary Optimization):** A metaheuristic that represents candidate solutions as quantum bits (Q-bits) with probability amplitudes, using quantum rotation gates to explore the search space more efficiently than classical evolutionary algorithms.
- **Feature Selection:** Reducing the dimensionality of extracted features to remove noise, reduce overfitting, and improve classifier performance.
- **Hyperparameter Optimization:** Searching for the best model hyperparameters (e.g., `C`, `gamma` for SVM; `learning_rate`, `max_depth` for XGBoost) that maximize cross-validated accuracy.

> **Note:** For full evaluation, comparison metrics, and statistical analysis, see **Notebook 06 — Evaluation & Comparison**.

---
## 1. Setup & Imports

We begin by importing all necessary modules from the project source tree. The `sys.path` insertion ensures that our custom `src/` package is importable when running from the `notebooks/` directory.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import json

# Data loading & feature extraction
from src.preprocessing.dataset import get_dataloaders
from src.features.feature_combiner import extract_and_combine_features

# Classical optimization
from src.optimization.classical_feature_selection import select_features_mi, select_features_kbest
from src.optimization.classical_hyperparam import optimize_svm_grid, optimize_xgboost_optuna

# Quantum-inspired optimization
from src.optimization.quantum_feature_selection import run_quantum_feature_selection
from src.optimization.quantum_hyperparam import run_quantum_hyperparam_opt

# Model training
from src.models.ml_classifier import train_and_evaluate_ml
from src.models.cnn_classifier import FineTunedResNet50, train_cnn

print('All imports successful.')

---
## 2. Load Data & Extract Features

We use our custom `get_dataloaders` function to load the raw medical imaging dataset (split into train / validation / test) and then extract a rich, combined feature vector from each image using our multi-extractor pipeline (ResNet50 deep features + radiomics + handcrafted texture descriptors).

The resulting `X_train` and `X_test` matrices are high-dimensional NumPy arrays that serve as input to all ML-based configurations (1, 2, 4, 5). Configuration 3 (CNN) operates directly on the image data loaders.

In [ ]:
# Load image data via PyTorch DataLoaders
train_loader, val_loader, test_loader, class_mapping = get_dataloaders('../data/raw', batch_size=16)

# Extract and combine multi-modal features
X_train, y_train, X_test, y_test = extract_and_combine_features(
    train_loader, test_loader, save_dir='../models'
)

print(f'Training set: {X_train.shape}, Test set: {X_test.shape}')
print(f'Classes: {class_mapping}')

---
## 3. Configuration 1 — Baseline A (SVM + Mutual Information)

### Strategy

**Feature Selection:** We use **Mutual Information (MI)** to rank all features by their statistical dependence with the target labels. MI captures both linear and non-linear relationships, making it a strong general-purpose filter method. We select the top `k=100` features.

**Hyperparameter Tuning:** A classical **GridSearchCV** exhaustively searches over a predefined grid of SVM hyperparameters:
- `C` (regularization strength): `[0.1, 1, 10, 100]`
- `gamma` (RBF kernel coefficient): `['scale', 'auto']`
- `kernel`: `['rbf']`

**Classifier:** Support Vector Machine with RBF kernel — a well-established baseline for medium-dimensional tabular data.

### Why This Baseline?

MI + GridSearchCV + SVM represents the "textbook" approach to feature-based medical image classification. It is deterministic, reproducible, and well-understood — making it an ideal reference point for measuring quantum improvements.

In [ ]:
# --- Configuration 1: Baseline A — SVM + Mutual Information ---

# Step 1: Feature selection via Mutual Information
X_train_mi, X_test_mi, mi_indices = select_features_mi(X_train, y_train, X_test, k=100)
print(f'Selected {len(mi_indices)} features via Mutual Information')

# Step 2: Hyperparameter optimization via GridSearchCV
best_svm_params = optimize_svm_grid(X_train_mi, y_train)
print(f'Best SVM params: {best_svm_params}')

# Step 3: Train and evaluate
train_and_evaluate_ml(
    X_train_mi, y_train, X_test_mi, y_test,
    classifier_type='svm',
    name='baseline_a_svm',
    class_names=list(class_mapping.keys()),
    svm_params=best_svm_params,
    save_dir='../models'
)

---
## 4. Configuration 2 — Baseline B (XGBoost + Optuna)

### Strategy

**Feature Selection:** **SelectKBest** with ANOVA F-value scoring. Unlike MI which measures general dependence, the F-test specifically captures linear discriminative power between classes. We again select the top `k=100` features, providing a different (but equally sized) feature subset for fair comparison.

**Hyperparameter Tuning:** **Optuna** with the Tree-structured Parzen Estimator (TPE) sampler — a Bayesian optimization approach that builds a probabilistic model of the objective function and intelligently proposes promising hyperparameter configurations. We run a small number of trials (`trials=3`) for demonstration; production runs typically use 50–200 trials.

The search space includes:
- `max_depth`: [3, 10]
- `learning_rate`: [0.01, 0.3]
- `n_estimators`: [50, 300]
- `subsample`: [0.6, 1.0]
- `colsample_bytree`: [0.6, 1.0]

**Classifier:** XGBoost — a gradient-boosted tree ensemble that excels on tabular data and is the de facto standard in many ML competitions.

### Why This Baseline?

This configuration represents a more modern and adaptive optimization pipeline (Bayesian optimization via Optuna) compared to the brute-force GridSearchCV in Baseline A. It establishes the upper bound of what classical Bayesian methods can achieve.

In [ ]:
# --- Configuration 2: Baseline B — XGBoost + Optuna ---

# Step 1: Feature selection via SelectKBest (ANOVA F-value)
X_train_kb, X_test_kb, kb_indices = select_features_kbest(X_train, y_train, X_test, k=100)
print(f'Selected {len(kb_indices)} features via SelectKBest')

# Step 2: Hyperparameter optimization via Optuna TPE
best_xgb_params = optimize_xgboost_optuna(X_train_kb, y_train, trials=3)
print(f'Best XGBoost params: {best_xgb_params}')

# Step 3: Train and evaluate
train_and_evaluate_ml(
    X_train_kb, y_train, X_test_kb, y_test,
    classifier_type='xgboost',
    name='baseline_b_xgb',
    class_names=list(class_mapping.keys()),
    xgb_params=best_xgb_params,
    save_dir='../models'
)

---
## 5. Configuration 3 — Baseline C (Fine-tuned ResNet50)

### Strategy

**Architecture:** A **ResNet50** model pre-trained on ImageNet, with the final fully-connected layer replaced to match our number of cancer classes. All layers are fine-tuned (not frozen) to allow the network to adapt its learned representations to the medical imaging domain.

**Training Details:**
- **Loss Function:** Focal Loss — a modification of cross-entropy that down-weights well-classified examples and focuses learning on hard, misclassified samples. This is critical for medical imaging where class imbalance is common.
- **Optimizer:** Adam with learning rate `1e-4`
- **Epochs:** 1 (for demonstration; production training uses 20–50 epochs with early stopping)

**Feature Selection / Hyperparameter Tuning:** Not applicable — the CNN learns features and classification boundaries end-to-end from raw pixel data.

### Why This Baseline?

Deep learning (specifically transfer learning with ResNet50) represents the state-of-the-art in medical image classification. It serves as a strong upper-bound reference: if quantum-optimized ML classifiers can approach or exceed CNN performance with far fewer parameters and more interpretable features, that is a significant result.

In [ ]:
# --- Configuration 3: Baseline C — Fine-tuned ResNet50 ---

# Train the CNN end-to-end
train_cnn(
    train_loader,
    val_loader,
    epochs=1,
    lr=0.0001,
    save_dir='../models'
)
print('ResNet50 baseline training complete.')

---
## 6. Configuration 4 — Quantum A (QIEO Feature Selection + SVM)

### Strategy

This is the first quantum-enhanced configuration. We replace the classical Mutual Information feature selection from Baseline A with our **Quantum-Inspired Evolutionary Optimization (QIEO)** algorithm.

### How QIEO Feature Selection Works

1. **Q-bit Representation:** Each candidate solution is a binary feature mask of length `d` (the total number of features). Instead of storing deterministic 0/1 values, each position is represented as a Q-bit with probability amplitudes `(α, β)` where `|α|² + |β|² = 1`. The probability of selecting feature `i` is `|β_i|²`.

2. **Population Initialization:** A population of `pop_size` Q-bit individuals is initialized with equal superposition: `α = β = 1/√2`, meaning each feature has a 50% chance of being selected.

3. **Measurement (Collapse):** Each Q-bit is "measured" by sampling from its probability distribution, producing a concrete binary feature mask.

4. **Fitness Evaluation:** The selected features are used to train a lightweight classifier (e.g., SVM with default params), and cross-validated accuracy serves as the fitness score.

5. **Quantum Rotation Gates:** After evaluating all individuals, quantum rotation gates are applied to update the probability amplitudes. The rotation angle `Δθ` is determined by comparing each individual to the current best solution:
   - If the best solution has feature `i` selected and the current individual does not, rotate towards `|1⟩`
   - If the best solution does not have feature `i` and the current individual does, rotate towards `|0⟩`
   - The magnitude of rotation is proportional to the fitness difference

6. **Iteration:** Steps 3–5 repeat for `max_iter` generations, progressively converging to a high-quality feature subset.

### Why Quantum Feature Selection?

Classical filter methods (MI, ANOVA) evaluate features independently — they cannot capture feature interactions. Wrapper methods (recursive elimination) are computationally expensive. QIEO provides a middle ground: it evaluates feature subsets holistically (like a wrapper) but explores the search space efficiently using quantum-inspired probability amplitudes (avoiding the exponential cost of exhaustive search).

In [ ]:
# --- Configuration 4: Quantum A — QIEO Feature Selection + SVM ---

# Step 1: Quantum-inspired feature selection
selected_indices = run_quantum_feature_selection(
    X_train, y_train,
    max_iter=5,
    pop_size=5,
    save_dir='../models'
)
print(f'QIEO selected {len(selected_indices)} features')

# Step 2: Apply the quantum-selected feature mask
X_train_q = X_train[:, selected_indices]
X_test_q = X_test[:, selected_indices]

# Step 3: Hyperparameter optimization via GridSearchCV (same as Baseline A)
best_svm_params_q = optimize_svm_grid(X_train_q, y_train)

# Step 4: Train and evaluate
train_and_evaluate_ml(
    X_train_q, y_train, X_test_q, y_test,
    classifier_type='svm',
    name='quantum_a_svm',
    class_names=list(class_mapping.keys()),
    svm_params=best_svm_params_q,
    save_dir='../models'
)

---
## 7. Configuration 5 — Quantum B (QIEO Hyperparameter Optimization + XGBoost)

### Strategy

This configuration combines the **QIEO-selected features** from Configuration 4 with **QIEO-based hyperparameter optimization** for XGBoost — making it the most fully quantum-optimized ML pipeline.

### How QIEO Hyperparameter Optimization Works

The key challenge is that hyperparameters are continuous or discrete values (e.g., `learning_rate ∈ [0.01, 0.3]`), not binary. QIEO solves this via **binary encoding of continuous parameters:**

1. **Binary Encoding:** Each hyperparameter is represented as a fixed-length binary string. For example, `learning_rate` might use 8 bits, giving 256 possible discrete values uniformly spaced in `[0.01, 0.3]`. All hyperparameters are concatenated into a single binary chromosome.

2. **Q-bit Representation:** Just like feature selection, each bit position has probability amplitudes `(α, β)`. Measuring the Q-bit string produces a concrete binary chromosome, which is decoded into real-valued hyperparameters.

3. **Fitness Evaluation:** The decoded hyperparameters are used to train an XGBoost model, and cross-validated accuracy serves as the fitness score.

4. **Quantum Rotation Gates:** Same update mechanism as feature selection — rotation angles are determined by comparing each individual's chromosome to the best-so-far solution.

### Hyperparameters Optimized

| Parameter | Range | Bits |
|-----------|-------|------|
| `learning_rate` | [0.01, 0.3] | 8 |
| `max_depth` | [3, 10] | 4 |
| `n_estimators` | [50, 300] | 9 |
| `subsample` | [0.6, 1.0] | 6 |
| `colsample_bytree` | [0.6, 1.0] | 6 |

### Why Quantum Hyperparameter Optimization?

Classical methods like GridSearchCV scale exponentially with the number of hyperparameters, and Bayesian methods (Optuna) can get stuck in local optima. QIEO maintains a population of solutions in quantum superposition, enabling broader exploration of the search space while still converging efficiently via rotation gate updates.

In [ ]:
# --- Configuration 5: Quantum B — QIEO Hyperparameter Optimization + XGBoost ---

# Step 1: Quantum-inspired hyperparameter optimization
best_params_q = run_quantum_hyperparam_opt(
    X_train_q, y_train,
    max_iter=5,
    pop_size=5,
    save_dir='../models'
)
print(f'QIEO optimized params: {best_params_q}')

# Step 2: Train and evaluate with quantum-optimized hyperparameters
train_and_evaluate_ml(
    X_train_q, y_train, X_test_q, y_test,
    classifier_type='xgboost',
    name='quantum_b_xgb',
    class_names=list(class_mapping.keys()),
    xgb_params=best_params_q,
    save_dir='../models'
)

---
## 8. Summary

We have now trained all **6 model configurations** for the QT-2.21 Quantum-Enhanced Medical Imaging project:

### Classical Baselines

| Config | Pipeline | Purpose |
|--------|----------|---------|
| **Baseline A** | MI Feature Selection → GridSearchCV → SVM | Classical filter + exhaustive search |
| **Baseline B** | SelectKBest → Optuna TPE → XGBoost | Classical filter + Bayesian search |
| **Baseline C** | Raw Images → Fine-tuned ResNet50 | Deep learning reference |

### Quantum-Optimized Models

| Config | Pipeline | Purpose |
|--------|----------|---------|
| **Quantum A** | QIEO Feature Selection → GridSearchCV → SVM | Quantum feature selection vs MI |
| **Quantum B** | QIEO Features → QIEO Hyperparam Opt → XGBoost | Fully quantum-optimized ML |
| **Quantum C** | Raw Images → Fine-tuned ResNet50 | Same as Baseline C (CNN reference) |

### Key Observations

- **Feature selection impact:** Compare the number and quality of features selected by MI/KBest vs QIEO. QIEO may select fewer features while maintaining or improving accuracy.
- **Hyperparameter quality:** Compare the XGBoost parameters found by Optuna vs QIEO. Both methods should find competitive configurations, but QIEO may converge faster.
- **Overall accuracy:** The ultimate comparison is test-set accuracy, precision, recall, and F1-score across all 6 configurations.

### Next Steps

Proceed to **Notebook 06 — Evaluation & Comparison** for:
- Side-by-side performance comparison (accuracy, precision, recall, F1, AUC-ROC)
- Confusion matrices for each configuration
- Statistical significance testing (McNemar's test, paired t-test)
- Visualization of quantum optimization convergence curves
- Final recommendations for clinical deployment